# 1.1 研究背景

**配對交易**：找價格長期同步的兩檔證券，價差偏離時放空高估、買進低估，回歸時平倉。
部位一多一空且金額相當 → 市場暴露趨近於零，屬**市場中性**統計套利。

- **GGR (2006)** 以 1962–2002 美股建立學術基準，報告扣成本後約 11% 年化超額報酬
- 其「形成期選對、交易期進出」兩階段架構成為後續文獻的共同基礎

**但獲利正在衰減**（Do & Faff 2010, 2012）：

| 機制 | 說明 |
| :--- | :--- |
| 套利資本增加 | 機會被快速消化 |
| 電子交易普及 | 發現價差的門檻降低 |
| 市場效率改善 | 資訊傳播加速 |

## 文獻的兩條回應路線

| 階段 | 改良方向 | 代表 |
| :--- | :--- | :--- |
| **形成期** | 改良配對的**尋找方式** | 共整合（Vidyamurthy 2004）、Copula（Liew & Wu 2013）、非監督分群（Sarmento & Horta 2020；Han et al. 2021） |
| **交易期** | 改良交易的**執行規則** | 隨機控制（Elliott et al. 2005）、Kalman、深度強化學習（Kim & Kim 2019） |

### 本研究聚焦交易期——理由是方法論的

> 交易期的改良可在**完全相同的一批配對**上施行，
> 兩臂承受相同的市場風險、配對品質與期間 → **天然的單變因設計**。
>
> 形成期無此便利：更換分群方法會**同時**改變候選池、配對品質與期間。

## 形成期在本研究中是「配對來源」

不作為被檢定的處理，而是提供**五個配對來源**，
在每一個來源上重複同一組交易端對照 → 結論不依賴單一來源。

- 不分組（NOGRP）
- GICS 產業分組
- HDBSCAN／Agglomerative／K-means（三種資料驅動分群）

五個來源**本身的優劣**列為描述性觀察（第五章 5.3），不作統計宣稱。

# 1.2 研究動機：一個未被檢視的設計選擇

DRL 應用於配對交易的理論吸引力：不需預測價格，
只需在給定狀態下**做出決策**，與「狀態—動作—報酬」框架天然契合。

> 然而**「做出什麼決策」——即動作空間（action space）的設定——
> 在既有文獻中是各自選定、未被比較的。**

## 兩種動作空間：126 次 vs 1 次

| 動作空間 | 每個交易期的決策次數 | 動作集合 |
| :--- | :--- | :--- |
| **逐日自由持倉** | **≤ 126 次** | {做多價差, 做空價差, 空手}，每日一次 |
| **每期單次選門檻** | **1 次** | SKIP ＋ 8 組 $(entry\_z, exit\_z)$ |

決策次數相差**兩個數量級**，而文獻通常只採用其中一種並報告績效。

- Kim & Kim (2019)：門檻選擇式（離散門檻組合）
- 朱羿璁 (2025)：共整合避險比率 ＋ 混合獎勵
- 另有一系列實作採**逐日決定持倉**

## 沒有研究把動作空間當唯一變因

**於是一個基本問題沒有答案**：

> 當學習式交易端失敗時，失敗來自**學習演算法**，還是來自**動作空間的設定**？

三項後果：

1. 失敗時無法歸因
2. 成功時無法區分增益來自「逐對辨識能力」或「動作空間的形狀」
3. 各研究頻率與母體不同 → 跨研究比較**無法替代**同一管線內的消融

## 這個問題在開發過程中以具體形式出現

本研究先後實作**三代逐日自由持倉**的交易端：

| 代 | 演算法 |
| :--- | :--- |
| v1 | online DQN（LSTM 編碼狀態） |
| v2 | v1 的修復版（修掉計算與統計缺陷） |
| v3 | FQI（批次擬合 Q） |

三代皆顯著劣於固定門檻基準，**且逐一修復訓練缺陷後劣勢依然存在**。

> 但開發過程中的觀察**不構成論文證據**：當時三代與現行交易端並非跑在
> 同一批配對上，且該批回測明細已於資料庫重建時遺失。
>
> → 本研究將它重做為一項**預先登記的受控消融**（第三章 3.3、第四章 4.1）。

## 本研究沿用的動作空間，其來源是日內研究

Kim & Kim (2019) 的結果建立於**日內**資料，
而本研究在**日頻**上檢定同一種動作空間設計。

> 這不是巧合而是本研究的處境：第二章 2.4 節將指出，
> 文獻中 AI 於配對交易的成功案例，其資料頻率**高度一致地集中於日內**。
>
> 故本研究的結果應讀為「該動作空間設計**在日頻上**的效果」，
> 而非對其原始設定的否證。

## 縮小動作空間之後，還剩下什麼能力？

門檻選擇式的交易端可以有兩種來源的增益：

| 來源 | 內容 | 實務意涵 |
| :--- | :--- | :--- |
| **逐對辨識能力** | 依每組配對的狀態選出適合它的門檻 | 指向更好的特徵與模型 |
| **動作空間的形狀** | 選單涵蓋的門檻水準 ＋「拒絕交易」的選項 | 指向更好的選單設計 |

**既有文獻報告學習式交易端優於固定規則時，通常未區分兩者。**

## 1.2.3 三個研究問題

> **Q1** 在其他條件完全相同時，逐日自由持倉的動作空間是否系統性造成
> 過度交易與績效劣化？（動作空間是否為關鍵約束）
>
> **Q2** 縮至「每期單次選門檻」後，逐對辨識能力是否存在——
> 即依配對狀態選門檻，是否優於選一個**最佳常數**？
>
> **Q3** 門檻選擇式相對固定門檻的效果有多大、在多少個配對來源上成立，
> 且該效果是否足以構成**可交易**的策略？

### Q3 的後半是刻意分開的界線

**「相對於某個基準較優」與「本身可交易」是兩個不同的主張**——
前者只需一個對照組，後者需要絕對績效、多重測試校正與成本餘裕。
本研究對兩者**分別檢定、分別報告**。

# 1.3 研究目的：三段證據鏈

以 S&P 500 成分股 2000–2025 年**日頻**資料，在受控的單變因設計下
檢驗**動作空間的設計對學習式配對交易端的影響**。

| 段 | 問題 | 內容 |
| :--- | :--- | :--- |
| 一 | Q1 | 動作空間消融：三代逐日自由持倉 vs 每期單次選門檻 vs 固定門檻 |
| 二 | Q2 | 逐對門檻學習 vs 擴張窗最佳常數 |
| 三 | Q3 | 門檻選擇式於五個配對來源上的效果與其絕對可交易性 |

三段的共同設計原則是**單一變因**——以**中性組裝器**架構
（特徵 → 分組 → 篩選 → 排序，四層可獨立替換）實現。

## 三項方法論目標

1. **推論基礎的正確性**——全網格等權組合為報告口徑（避免選擇偏誤）；
   逐日報酬差 ＋ 循環 block bootstrap；同時報 $p$ 值與信賴區間；BH 校正
2. **判準先於資料**——消融採**預先登記**，假說／門檻／核對程序皆於
   回測啟動前提交版本庫；偏離逐筆記於附錄 D，
   **包含一處由研究者自身實作錯誤所致、更正後使核對由失敗轉為通過的情形**
3. **限制與缺陷的量化揭露**——存活者偏誤、覆蓋率、成本假設年代皆以
   具體數字與偏誤方向呈現；實作缺陷列附錄 B，不因已修正而省略

# 1.4 研究架構

| 章 | 內容 |
| :--- | :--- |
| 一 緒論 | 動作空間是未被檢視的設計選擇；Q1–Q3 |
| 二 文獻探討 | 以**動作空間**為軸重讀交易端文獻；AI 成功案例的頻率相依 |
| 三 研究方法（系統設計） | 五個配對來源；**動作空間的設計空間**；DL-THR／RL-THR；評估與預先登記 |
| 四 實證結果 | 依三段證據鏈：消融 → 最佳常數對照 → 主檢定 → 替代解釋 → 穩健性 |
| 五 討論 | 為何統計顯著仍不可交易（三道界線）；損益結構；來源的描述性觀察；限制 |
| 六 結論 | 發現綜整、貢獻、建議、結語 |

**附錄**　A 前行研究差異定位｜B 實作缺陷記錄｜C 交易端改良的受控否證｜**D 動作空間消融的預先登記**

## 本章小結

- 配對交易的獲利證據建立於 2000 年代之前，後續研究一致指出其**衰減**
- 學習式交易端被提出作為交易期的改良，但既有文獻**各自選定一種動作空間**
  並報告績效，未有研究在同一管線內把它當作唯一變因加以消融
- 於是「學習式交易端為何成功或失敗」缺少乾淨的歸因

> 本研究的三個問題答案**並不一致**，而這種不一致本身就是發現：
> **動作空間的設計確實關鍵，逐對辨識能力則未獲支持，
> 而相對效果雖在多個來源上成立，卻不足以跨過可交易的門檻。**